In [ ]:
import json
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns
import ipywidgets as widgets
from IPython.display import display, Image, clear_output

import sys
from pathlib import Path

ROOT = Path.cwd().parents[1]
sys.path.insert(0, str(ROOT))

from src.config.paths import *
from src.data.plots import plot_zero_boxes

ANALYSIS_DIR = CSV_DIR / "data_analysis"
IMAGE_DIR = IMAGE_DIR / "data_analysis"
PLOT_DIR = PLOT_DIR / "data_analysis"

In [ ]:
raw_df = pd.read_csv(CSV_DIR/"data/metadata_raw.csv")

raw_df["boxes"] = raw_df["boxes"].apply(json.loads)

In [ ]:
raw_sizes = raw_df["boxes"].explode().dropna().apply(pd.Series)

raw_sizes[["xc", "yc", "width", "height"]] = pd.DataFrame(
    raw_sizes["coordinates"].tolist(),
    index=raw_sizes.index
)

raw_sizes["path"] = raw_df.loc[raw_sizes.index, "path"].to_numpy()
raw_sizes["area"] = raw_sizes["width"] * raw_sizes["height"]
raw_sizes["split"] = raw_df.loc[raw_sizes.index, "split"].to_numpy()

raw_sizes

In [ ]:
zero_boxes = raw_sizes[raw_sizes["area"] == 0]

print("Zero-area boxes:", len(zero_boxes))

In [ ]:
x_min = (raw_sizes["xc"] - raw_sizes["width"] / 2) < 0
x_max = (raw_sizes["xc"] + raw_sizes["width"] / 2) > 1

y_min = (raw_sizes["yc"] - raw_sizes["height"] / 2) < 0
y_max = (raw_sizes["yc"] + raw_sizes["height"] / 2) > 1

out_of_bounds = x_min | x_max | y_min | y_max

print("Total out-of-bounds boxes:", out_of_bounds.sum())
print("Percentage:", f"{out_of_bounds.mean() * 100:.2f}%")
print("Affected images:", raw_sizes.loc[out_of_bounds, "path"].nunique())

In [ ]:
bounds = pd.DataFrame({
    "left": raw_sizes["xc"] - raw_sizes["width"] / 2,
    "right": raw_sizes["xc"] + raw_sizes["width"] / 2,
    "top": raw_sizes["yc"] - raw_sizes["height"] / 2,
    "bottom": raw_sizes["yc"] + raw_sizes["height"] / 2
})

# Treats values as > 0 only clips when < 0
# For example [-0.15, 0.8, 0.1, -0.2] -> [0.15, 0, 0, 0.2]
violations = pd.DataFrame({
    "left": (-bounds["left"]).clip(lower=0),
    "right": (bounds["right"] - 1).clip(lower=0),
    "top": (-bounds["top"]).clip(lower=0),
    "bottom": (bounds["bottom"] - 1).clip(lower=0)
})

raw_sizes["boundary_excess"] = violations.max(axis=1)
raw_sizes.loc[out_of_bounds, "boundary_excess"].describe(
    percentiles=[0.5, 0.9, 0.95, 0.99]
)

In [ ]:
image_paths = plot_zero_boxes(zero_boxes, raw_df)
current = 0

prev_btn = widgets.Button(description="Previous")
next_btn = widgets.Button(description="Next")
output = widgets.Output()

def show_image():
    with output:
        clear_output(wait=True)
        display(Image(filename=str(image_paths[current])))

    prev_btn.disabled = current == 0
    next_btn.disabled = current == len(image_paths) - 1

def navigate(change):
    global current

    if change == next_btn:
        current += 1
    else:
        current -= 1

    show_image()

prev_btn.on_click(navigate)
next_btn.on_click(navigate)

display(widgets.HBox([prev_btn, next_btn]), output)
show_image()

In [ ]:
sizes_df = pd.read_csv(ANALYSIS_DIR/"box_sizes.csv")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

plot_df = sizes_df.copy()
plot_df["category"] = plot_df["category"].map({
    0: "Smoke",
    1: "Fire"
})

sns.kdeplot(
    data=plot_df,
    x="area",
    hue="category",
    hue_order=["Smoke", "Fire"],
    palette={"Smoke": "grey", "Fire": "red"},
    common_norm=False,
    fill=True,
    alpha=0.25,
    linewidth=2,
    cut=0,
    bw_adjust=3, # Makes the curve smoother
    ax=ax
)

ax.set_title("Distribution of Fire and Smoke Bounding-Box Areas")
ax.set_xlabel("Relative Bounding-Box Area")
ax.set_ylabel("Probability Density")

fig.tight_layout()
fig.savefig(PLOT_DIR / "fire_vs_smoke_area_distribution.svg")
plt.show()

Probability density represents how concentrated bounding-box areas are within a particular range, where the probability of an annotation falling within that range is represented by the area under the curve.

For example:
Relative bounding box-area: 0.05
Probability Density: 12

Probability = 12 x 0.05 = 0.06 => 6%

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

sns.kdeplot(
    data=sizes_df,
    x="area",
    color="tab:blue",
    fill=True,
    alpha=0.25,
    linewidth=2,
    cut=0,
    ax=ax
)

ax.set_title("Distribution of Bounding-Box Areas")
ax.set_xlabel("Relative Bounding-Box Area")
ax.set_ylabel("Probability Density")

fig.tight_layout()
fig.savefig(PLOT_DIR / "box_area_distribution.svg")
plt.show()

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(18, 10))

for ax, split in zip(axes, ["train", "val", "test"]):
    ax.boxplot(
        [
            sizes_df["area"][(sizes_df["split"] == split) & (sizes_df["category"] == 1)],
            sizes_df["area"][(sizes_df["split"] == split) & (sizes_df["category"] == 0)],
            sizes_df["area"][sizes_df["split"] == split],
        ],
        orientation="horizontal",
        tick_labels=["Fire", "Smoke", "All boxes"],
        showfliers=False
    )
    ax.set_xticks(np.arange(0, 1.1, 0.1))
    ax.grid(axis="x", linestyle="--", alpha=0.5)
    ax.set_title(split)

plt.savefig(PLOT_DIR / "area_dist_by_split.svg")
plt.show()